# Optuna Tuning and MLflow Tracking


In [1]:
import hashlib
import inspect
import importlib.metadata
import json
import os
import sys
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import numpy as np
import optuna
import pandas as pd
import xgboost as xgb
from mlflow.tracking import MlflowClient
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "data" / "processed" / "Charging_Data_clean.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Could not find data/processed/Charging_Data_clean.csv")

TEAM_DIR = ROOT / "team" / "mariam_availability"
sys.path.insert(0, str(TEAM_DIR))
from src.availability.features import FeatureEncoder, split_by_time

ARTIFACTS = TEAM_DIR / "artifacts"
REPORTS = TEAM_DIR / "reports"
REPORTS.mkdir(parents=True, exist_ok=True)
CLEAN_PATH = ROOT / "data" / "processed" / "Charging_Data_clean.csv"
FEATURES_PATH = ARTIFACTS / "features_all.pkl"
ENCODER_PATH = ARTIFACTS / "encoder_state.pkl"
CONFIG_PATH = ARTIFACTS / "config_split.pkl"
TARGET_MODE = "log1p"
RANDOM_STATE = 42
EXPERIMENT_NAME = "chargesentinel-availability"
MODEL_NAME = "availability_duration"

print("Optuna:", importlib.metadata.version("optuna"))
print("MLflow:", importlib.metadata.version("mlflow"))
print("XGBoost:", importlib.metadata.version("xgboost"))

Optuna: 5.0.0
MLflow: 3.17.2
XGBoost: 3.4.1


## 1. Load the cleaning and feature outputs



In [2]:
with FEATURES_PATH.open("rb") as handle:
    features = pd.read_pickle(handle)
with ENCODER_PATH.open("rb") as handle:
    encoder_state = joblib.load(handle)
with CONFIG_PATH.open("rb") as handle:
    config = joblib.load(handle)

TRAIN_END = pd.Timestamp(config["TRAIN_END"])
VAL_END = pd.Timestamp(config["VAL_END"])
CAP = float(config["CAP"])
train, validation, test = split_by_time(features)
train_normal = train.loc[train.is_abnormal == 0].copy()
validation_normal = validation.loc[validation.is_abnormal == 0].copy()
encoder = FeatureEncoder.from_state(encoder_state)
X_train = encoder.transform(train_normal)
X_validation_normal = encoder.transform(validation_normal)
X_validation = encoder.transform(validation)
X_test = encoder.transform(test)
FEATURE_NAMES = list(encoder_state["feature_names"])


def to_target(values):
    values = np.minimum(np.asarray(values, dtype=float), CAP)
    return np.log1p(values) if TARGET_MODE == "log1p" else values


def from_target(values):
    values = np.asarray(values, dtype=float)
    return np.expm1(values) if TARGET_MODE == "log1p" else values


def data_version(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return {
        "data_file": Path(path).name,
        "data_sha256_12": digest.hexdigest()[:12],
        "n_rows": len(features),
        "train_end": str(TRAIN_END),
        "val_end": str(VAL_END),
    }


DATA_VERSION = data_version(CLEAN_PATH)
clean_rows = sum(1 for _ in CLEAN_PATH.open(encoding="utf-8")) - 1
print(f"Clean rows / feature rows: {clean_rows:,} / {len(features):,}")
print(f"Train / validation / test: {len(train):,} / {len(validation):,} / {len(test):,}")
print(f"Normal training rows: {len(train_normal):,}; CAP: {CAP:.3f} minutes")
print("Clean CSV SHA-256 (12 chars):", DATA_VERSION["data_sha256_12"])
assert clean_rows == len(features)
assert all(not matrix.isna().any().any()
           for matrix in [X_train, X_validation_normal, X_validation, X_test])

Clean rows / feature rows: 441,064 / 441,064
Train / validation / test: 285,225 / 75,767 / 80,072
Normal training rows: 233,397; CAP: 565.287 minutes
Clean CSV SHA-256 (12 chars): e518e9c6d140


## 2. Configure MLflow


In [3]:
ARTIFACTS.mkdir(parents=True, exist_ok=True)
TRACKING_DB = ARTIFACTS / "mlflow.db"
LOCAL_MLRUNS = ARTIFACTS / "mlruns"
SHARED_TRACKING_URI = os.getenv("CHARGESENTINEL_MLFLOW_TRACKING_URI")
TRACKING_URI = SHARED_TRACKING_URI or ("sqlite:///" + TRACKING_DB.as_posix())
mlflow.set_tracking_uri(TRACKING_URI)

experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    if SHARED_TRACKING_URI:
        experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
    else:
        experiment_id = mlflow.create_experiment(
            EXPERIMENT_NAME,
            artifact_location=LOCAL_MLRUNS.resolve().as_uri(),
        )
    experiment = mlflow.get_experiment(experiment_id)
mlflow.set_experiment(EXPERIMENT_NAME)
EXPERIMENT_ID = experiment.experiment_id

COMMON_TAGS = {
    **{key: str(value) for key, value in DATA_VERSION.items()},
    "feature_version": "start_known_features_v1",
    "clean_rows": str(len(features)),
    "test_set_previously_scored": "true",
    "prior_test_metrics_data_rows": "440972",
}


def set_run_tags(model_family, split_role="validation"):
    mlflow.set_tags({**COMMON_TAGS, "model_family": model_family, "split_role": split_role})


def log_feature_list():
    mlflow.log_dict({"feature_names": FEATURE_NAMES}, "feature_list.json")


def metrics_for(frame, predictions, split_name):
    actual = frame.target_min.to_numpy(dtype=float)
    errors = np.asarray(predictions, dtype=float) - actual
    normal_errors = errors[frame.is_abnormal.to_numpy() == 0]
    return {
        f"{split_name}_mae_normal": float(np.mean(np.abs(normal_errors))),
        f"{split_name}_rmse_normal": float(np.sqrt(np.mean(normal_errors ** 2))),
        f"{split_name}_median_ae_normal": float(np.median(np.abs(normal_errors))),
        f"{split_name}_mae_all": float(np.mean(np.abs(errors))),
    }


def log_model_compat(log_model, model, example, trusted_types=None):
    parameters = inspect.signature(log_model).parameters
    path_parameter = "name" if "name" in parameters else "artifact_path"
    log_kwargs = {
        path_parameter: "model",
        "input_example": example,
    }
    if trusted_types and "skops_trusted_types" in parameters:
        log_kwargs["skops_trusted_types"] = trusted_types
    return log_model(model, **log_kwargs)


def log_xgb_model(model, example):
    return log_model_compat(mlflow.xgboost.log_model, model, example)


def log_rf_model(model, example):
    return log_model_compat(
        mlflow.sklearn.log_model,
        model,
        example,
        trusted_types=["sklearn.tree._tree.Tree"],
    )

def log_importance(model, title):
    importance = pd.Series(model.feature_importances_, index=FEATURE_NAMES).nlargest(20).sort_values()
    fig, ax = plt.subplots(figsize=(9, 6))
    importance.plot.barh(ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Feature importance")
    fig.tight_layout()
    mlflow.log_figure(fig, "feature_importance.png")
    plt.close(fig)


print("MLflow URI:", TRACKING_URI)
print("Experiment:", EXPERIMENT_NAME)

MLflow URI: sqlite:///c:/Users/mariam/Desktop/SIC AI/ChargeSentinel-Twin/team/mariam_availability/artifacts/mlflow.db
Experiment: chargesentinel-availability


## 3. Log the median baseline

The baseline has no fitted model or feature-importance vector. Its run logs the median lookup tables and a profile chart instead.

In [4]:
def hour_bucket(hours):
    return pd.cut(hours, [-1, 5, 10, 14, 18, 23], labels=False)


baseline_train = train_normal.copy()
baseline_train["hour_bucket"] = hour_bucket(baseline_train.hour)
median_by_charger_hour = baseline_train.groupby(
    ["charger_id", "hour_bucket"]
).target_min.median().to_dict()
median_by_charger = baseline_train.groupby("charger_id").target_min.median().to_dict()
median_by_location = baseline_train.groupby("location").target_min.median().to_dict()
global_median = float(baseline_train.target_min.median())


def predict_baseline(frame):
    predictions = []
    for charger_id, bucket, location in zip(
        frame.charger_id, hour_bucket(frame.hour), frame.location
    ):
        prediction = median_by_charger_hour.get(
            (charger_id, bucket),
            median_by_charger.get(charger_id, median_by_location.get(location, global_median)),
        )
        predictions.append(prediction)
    return np.asarray(predictions, dtype=float)


baseline_validation_predictions = predict_baseline(validation)
baseline_validation_metrics = metrics_for(
    validation, baseline_validation_predictions, "validation"
)
baseline_lookup = {
    "by_charger_hour": [
        {"charger_id": str(charger), "hour_bucket": int(bucket), "median_duration_min": float(value)}
        for (charger, bucket), value in median_by_charger_hour.items()
    ],
    "by_charger": {str(key): float(value) for key, value in median_by_charger.items()},
    "by_location": {str(key): float(value) for key, value in median_by_location.items()},
    "global_median": global_median,
}

with mlflow.start_run(run_name="baseline"):
    mlflow.log_params({
        "method": "median_by_charger_hour_bucket_with_fallbacks",
        "target_mode": TARGET_MODE,
        "CAP_minutes": CAP,
        "hour_bucket_edges": "-1,5,10,14,18,23",
    })
    set_run_tags("baseline")
    mlflow.log_metrics(baseline_validation_metrics)
    log_feature_list()
    mlflow.log_dict(baseline_lookup, "baseline_lookup_tables.json")
    profile = baseline_train.groupby("hour_bucket").target_min.median()
    fig, ax = plt.subplots(figsize=(8, 4))
    profile.plot.bar(ax=ax)
    ax.set_title("Baseline median duration by hour bucket")
    ax.set_ylabel("Median duration (minutes)")
    fig.tight_layout()
    mlflow.log_figure(fig, "baseline_profile.png")
    plt.close(fig)

print("Baseline validation:", baseline_validation_metrics)

Baseline validation: {'validation_mae_normal': 44.107765569911194, 'validation_rmse_normal': 109.29421800474935, 'validation_median_ae_normal': 19.200000000000003, 'validation_mae_all': 43.06688399963045}


## 4. Refit Random Forest and default XGBoost

Both use the same corrected training rows, CAP, target transformation, and validation partitions. Only validation scores enter the model comparison.

In [5]:
RF_PARAMS = {
    "n_estimators": 200,
    "min_samples_leaf": 5,
    "max_features": 0.5,
    "max_samples": 0.5,
    "n_jobs": -1,
    "random_state": 42,
}
rf_model = RandomForestRegressor(**RF_PARAMS)
rf_model.fit(X_train, to_target(train_normal.target_min))
rf_validation_predictions = from_target(rf_model.predict(X_validation))
rf_validation_metrics = metrics_for(validation, rf_validation_predictions, "validation")

with mlflow.start_run(run_name="random_forest"):
    mlflow.log_params({**RF_PARAMS, "target_mode": TARGET_MODE, "CAP_minutes": CAP})
    set_run_tags("random_forest")
    mlflow.log_metrics(rf_validation_metrics)
    log_feature_list()
    log_importance(rf_model, "Random Forest feature importance")
    log_rf_model(rf_model, X_validation_normal.head(5))

print("Random Forest validation:", rf_validation_metrics)

C:\Users\mariam\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


Random Forest validation: {'validation_mae_normal': 43.628534927008445, 'validation_rmse_normal': 107.14628219759717, 'validation_median_ae_normal': 18.37921790806655, 'validation_mae_all': 40.80993478420318}


In [6]:
DEFAULT_XGB_PARAMS = {
    "n_estimators": 3000,
    "learning_rate": 0.05,
    "max_depth": 8,
    "min_child_weight": 5,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "tree_method": "hist",
    "n_jobs": -1,
    "random_state": 42,
    "early_stopping_rounds": 100,
    "eval_metric": "mae",
}
default_xgb_model = xgb.XGBRegressor(**DEFAULT_XGB_PARAMS)
default_xgb_model.fit(
    X_train,
    to_target(train_normal.target_min),
    eval_set=[(X_validation_normal, to_target(validation_normal.target_min))],
    verbose=False,
)
default_xgb_validation_predictions = from_target(default_xgb_model.predict(X_validation))
default_xgb_validation_metrics = metrics_for(
    validation, default_xgb_validation_predictions, "validation"
)

with mlflow.start_run(run_name="xgboost_default"):
    mlflow.log_params({**DEFAULT_XGB_PARAMS, "target_mode": TARGET_MODE, "CAP_minutes": CAP})
    set_run_tags("xgboost_default")
    mlflow.log_metrics(default_xgb_validation_metrics)
    mlflow.set_tag("best_iteration", str(default_xgb_model.best_iteration))
    log_feature_list()
    log_importance(default_xgb_model, "Default XGBoost feature importance")
    log_xgb_model(default_xgb_model, X_validation_normal.head(5))

print("Default XGBoost best iteration:", default_xgb_model.best_iteration)
print("Default XGBoost validation:", default_xgb_validation_metrics)

C:\Users\mariam\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


Default XGBoost best iteration: 92
Default XGBoost validation: {'validation_mae_normal': 43.40372372646231, 'validation_rmse_normal': 107.27574201982662, 'validation_median_ae_normal': 18.24126754167949, 'validation_mae_all': 40.692388592834206}


## 5. Create time-ordered folds

The feature table is stored in charger/start order, so sort normal training rows by start before applying TimeSeriesSplit. Refit the encoder inside each fold.

In [7]:
order = train_normal.sort_values("start").index.to_numpy()
time_split = TimeSeriesSplit(n_splits=3)
folds = []
for train_positions, valid_positions in time_split.split(order):
    train_indices = order[train_positions]
    valid_indices = order[valid_positions]
    folds.append((train_indices, valid_indices))

for fold_number, (train_indices, valid_indices) in enumerate(folds, start=1):
    fold_train = train_normal.loc[train_indices]
    fold_valid = train_normal.loc[valid_indices]
    assert fold_train.start.max() <= fold_valid.start.min()
    print(
        f"Fold {fold_number}: train through {fold_train.start.max()} | "
        f"validation {fold_valid.start.min()} to {fold_valid.start.max()}"
    )

Fold 1: train through 2020-07-22 16:19:12 | validation 2020-07-22 16:20:40 to 2020-10-29 23:44:15
Fold 2: train through 2020-10-29 23:44:15 | validation 2020-10-29 23:45:34 to 2021-03-04 14:11:17
Fold 3: train through 2021-03-04 14:11:17 | validation 2021-03-04 14:11:19 to 2021-06-30 23:51:03


## 6. Run the bounded Optuna search

The objective is average normal-session validation MAE across three expanding folds. Each trial is a nested MLflow run with sampled parameters, fold MAEs, and best iterations. Stop at 25 trials or 45 minutes.

In [8]:
def objective(trial):
    sampled_params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.2, log=True),
        "max_depth": trial.suggest_int("max_depth", 4, 10),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 10.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 1.0, log=True),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
    }
    fold_maes = []
    fold_iterations = []

    with mlflow.start_run(run_name=f"trial_{trial.number:03d}", nested=True):
        mlflow.log_params({
            **sampled_params,
            "trial_number": trial.number,
            "n_estimators": 2000,
            "early_stopping_rounds": 50,
            "n_splits": len(folds),
            "target_mode": TARGET_MODE,
            "CAP_minutes": CAP,
        })
        set_run_tags("xgboost_optuna_trial", split_role="time_ordered_cv")

        for fold_number, (train_indices, valid_indices) in enumerate(folds, start=1):
            fold_train = train_normal.loc[train_indices]
            fold_valid = train_normal.loc[valid_indices]
            fold_encoder = FeatureEncoder().fit(fold_train)
            X_fold_train = fold_encoder.transform(fold_train)
            X_fold_valid = fold_encoder.transform(fold_valid)
            y_fold_train = to_target(fold_train.target_min)
            y_fold_valid = to_target(fold_valid.target_min)

            fold_model = xgb.XGBRegressor(
                n_estimators=2000,
                early_stopping_rounds=50,
                tree_method="hist",
                n_jobs=-1,
                random_state=42,
                eval_metric="mae",
                **sampled_params,
            )
            fold_model.fit(
                X_fold_train,
                y_fold_train,
                eval_set=[(X_fold_valid, y_fold_valid)],
                verbose=False,
            )
            fold_predictions = from_target(fold_model.predict(X_fold_valid))
            fold_mae = float(np.mean(np.abs(
                fold_predictions - fold_valid.target_min.to_numpy()
            )))
            best_iteration = int(fold_model.best_iteration)
            fold_maes.append(fold_mae)
            fold_iterations.append(best_iteration)
            mlflow.log_metric(f"fold_{fold_number}_mae_minutes", fold_mae)
            mlflow.log_metric(f"fold_{fold_number}_best_iteration", best_iteration)

        mean_fold_mae = float(np.mean(fold_maes))
        median_iteration = int(np.median(fold_iterations))
        trial.set_user_attr("fold_maes", fold_maes)
        trial.set_user_attr("fold_best_iterations", fold_iterations)
        trial.set_user_attr("median_best_iteration", median_iteration)
        mlflow.log_metric("mean_cv_mae_minutes", mean_fold_mae)
        mlflow.log_metric("median_best_iteration", median_iteration)

    return mean_fold_mae


with mlflow.start_run(run_name="optuna_xgboost"):
    mlflow.log_params({
        "sampler": "TPESampler",
        "sampler_seed": 42,
        "n_splits": len(folds),
        "max_trials": 25,
        "timeout_seconds": 2700,
        "objective": "mean_time_ordered_fold_mae_minutes",
    })
    set_run_tags("optuna_study", split_role="training_period_cv")
    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=42),
    )
    study.optimize(
        objective,
        n_trials=25,
        timeout=45 * 60,
        gc_after_trial=True,
        show_progress_bar=True,
    )
    completed_trials = [
        t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE
    ]
    if not completed_trials:
        raise RuntimeError("Optuna completed no trials; no tuned model is available.")

    trials_path = REPORTS / "availability_xgboost_optuna_trials.csv"
    study.trials_dataframe(
        attrs=("number", "value", "params", "user_attrs", "state")
    ).to_csv(trials_path, index=False)
    study_path = ARTIFACTS / "availability_xgboost_optuna_study.joblib"
    joblib.dump(study, study_path)
    mlflow.log_artifact(str(trials_path), artifact_path="optuna")
    mlflow.log_artifact(str(study_path), artifact_path="optuna")
    mlflow.log_dict({
        "best_params": study.best_params,
        "best_cv_mae_minutes": study.best_value,
        "completed_trials": len(completed_trials),
        "timeout_seconds": 2700,
    }, "optuna_best_result.json")
    mlflow.log_metric("best_cv_mae_minutes", float(study.best_value))
    mlflow.log_metric("completed_trials", len(completed_trials))

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        optuna.visualization.matplotlib.plot_optimization_history(study)
        history_figure = plt.gcf()
        mlflow.log_figure(history_figure, "optuna_optimization_history.png")
        plt.close(history_figure)
        if len(completed_trials) >= 2:
            importance = optuna.importance.get_param_importances(study)
            importance_figure, importance_axis = plt.subplots(figsize=(8, 5))
            pd.Series(importance).sort_values().plot.barh(ax=importance_axis)
            importance_axis.set_title("Optuna parameter importance")
            importance_figure.tight_layout()
            mlflow.log_figure(importance_figure, "optuna_parameter_importance.png")
            plt.close(importance_figure)

print("Completed trials:", len(completed_trials))
print("Best mean fold MAE:", study.best_value)
print("Best parameters:", study.best_params)

[I 2026-10-28 04:12:22,427] A new study created in memory with name: no-name-e774f6b1-44ba-47a1-816a-9d1b964a77fb


  0%|          | 0/25 [00:00<?, ?it/s]

[I 2026-10-28 04:12:26,614] Trial 0 finished with value: 38.958262680136066 and parameters: {'learning_rate': 0.061053157936408836, 'max_depth': 10, 'min_child_weight': 15, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.5780093202212182, 'reg_lambda': 0.2051110418843398, 'reg_alpha': 0.0014936568554617625, 'gamma': 4.330880728874676}. Best is trial 0 with value: 38.958262680136066.
[I 2026-10-28 04:12:30,390] Trial 1 finished with value: 38.969930381542326 and parameters: {'learning_rate': 0.0938393881004349, 'max_depth': 8, 'min_child_weight': 1, 'subsample': 0.9879639408647978, 'colsample_bytree': 0.9162213204002109, 'reg_lambda': 0.26587543983272705, 'reg_alpha': 0.0035113563139704067, 'gamma': 0.9170225492671691}. Best is trial 0 with value: 38.958262680136066.
[I 2026-10-28 04:12:34,594] Trial 2 finished with value: 38.90945021011947 and parameters: {'learning_rate': 0.05343048274530412, 'max_depth': 7, 'min_child_weight': 9, 'subsample': 0.7164916560792167, 'colsample_byt

## 7. Refit tuned XGBoost and choose from validation

Tuned XGBoost wins only if normal-session validation MAE improves on default XGBoost by at least 1%. Otherwise, default XGBoost remains champion. The test partition is not consulted here.

In [9]:
TUNED_XGB_PARAMS = {
    **study.best_params,
    "n_estimators": 3000,
    "tree_method": "hist",
    "n_jobs": -1,
    "random_state": 42,
    "early_stopping_rounds": 100,
    "eval_metric": "mae",
}
tuned_xgb_model = xgb.XGBRegressor(**TUNED_XGB_PARAMS)
tuned_xgb_model.fit(
    X_train,
    to_target(train_normal.target_min),
    eval_set=[(X_validation_normal, to_target(validation_normal.target_min))],
    verbose=False,
)
tuned_xgb_validation_predictions = from_target(tuned_xgb_model.predict(X_validation))
tuned_xgb_validation_metrics = metrics_for(
    validation, tuned_xgb_validation_predictions, "validation"
)

with mlflow.start_run(run_name="xgboost_tuned"):
    mlflow.log_params({**TUNED_XGB_PARAMS, "target_mode": TARGET_MODE, "CAP_minutes": CAP})
    set_run_tags("xgboost_tuned")
    mlflow.log_metrics(tuned_xgb_validation_metrics)
    mlflow.set_tag("best_iteration", str(tuned_xgb_model.best_iteration))
    mlflow.log_metric("best_cv_mae_minutes", float(study.best_value))
    log_feature_list()
    log_importance(tuned_xgb_model, "Tuned XGBoost feature importance")
    log_xgb_model(tuned_xgb_model, X_validation_normal.head(5))

candidate_predictions = {
    "Baseline": baseline_validation_predictions,
    "Random Forest": rf_validation_predictions,
    "XGBoost default": default_xgb_validation_predictions,
    "XGBoost tuned": tuned_xgb_validation_predictions,
}
validation_rows = []
for model_name, predictions in candidate_predictions.items():
    validation_rows.append({
        "model": model_name,
        **metrics_for(validation, predictions, "validation"),
    })
validation_comparison = pd.DataFrame(validation_rows)
validation_path = REPORTS / "availability_duration_validation_comparison.csv"
validation_comparison.to_csv(validation_path, index=False)

default_mae = default_xgb_validation_metrics["validation_mae_normal"]
tuned_mae = tuned_xgb_validation_metrics["validation_mae_normal"]
tuned_relative_gain = (default_mae - tuned_mae) / default_mae
minimum_tuning_gain = 0.01
if tuned_relative_gain >= minimum_tuning_gain:
    champion_name = "xgboost_tuned"
    champion_model = tuned_xgb_model
    champion_params = TUNED_XGB_PARAMS
else:
    champion_name = "xgboost_default"
    champion_model = default_xgb_model
    champion_params = DEFAULT_XGB_PARAMS

selection_report = {
    "selection_metric": "normal-session validation MAE (minutes)",
    "minimum_tuned_gain_fraction": minimum_tuning_gain,
    "default_xgboost_validation_mae_normal": default_mae,
    "tuned_xgboost_validation_mae_normal": tuned_mae,
    "tuned_relative_gain_fraction": tuned_relative_gain,
    "champion": champion_name,
    "test_set_previously_scored_before_tuning": True,
    "prior_test_metrics_file": "notebooks/reports/duration_test_metrics.csv",
    "data_version": DATA_VERSION,
}
selection_path = REPORTS / "availability_duration_selection.json"
selection_path.write_text(json.dumps(selection_report, indent=2), encoding="utf-8")
print(validation_comparison.to_string(index=False))
print(f"Tuned relative validation MAE gain: {tuned_relative_gain:.2%}")
print("Selected from validation only:", champion_name)

C:\Users\mariam\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


          model  validation_mae_normal  validation_rmse_normal  validation_median_ae_normal  validation_mae_all
       Baseline              44.107766              109.294218                    19.200000           43.066884
  Random Forest              43.628535              107.146282                    18.379218           40.809935
XGBoost default              43.403724              107.275742                    18.241268           40.692389
  XGBoost tuned              43.397954              107.173239                    18.251621           40.664084
Tuned relative validation MAE gain: 0.01%
Selected from validation only: xgboost_default


## 8. Log and register the champion; score test once

Notebook 05 contains a prior test evaluation, but its saved outputs came from the earlier 440,972-row feature snapshot. Treat those metrics as historical and do not compare them directly with this corrected 441,064-row data version. This notebook selects the champion using validation only, then scores the current held-out test partition once and records that prior test exposure.

In [10]:
bundle_path = ARTIFACTS / "availability_duration_bundle_optuna_mlflow.joblib"
bundle = {
    "model": champion_model,
    "model_name": champion_name,
    "encoder_state": encoder_state,
    "feature_names": FEATURE_NAMES,
    "target_mode": TARGET_MODE,
    "CAP": CAP,
    "split_dates": {"TRAIN_END": str(TRAIN_END), "VAL_END": str(VAL_END)},
    "data_version": DATA_VERSION,
    "baseline": baseline_lookup,
    "validation_metrics": validation_comparison.to_dict("records"),
}
joblib.dump(bundle, bundle_path)

with mlflow.start_run(run_name="champion") as champion_run:
    mlflow.log_params({
        **champion_params,
        "selected_model": champion_name,
        "target_mode": TARGET_MODE,
        "CAP_minutes": CAP,
        "minimum_tuned_gain_fraction": minimum_tuning_gain,
    })
    set_run_tags(champion_name, split_role="validation_selected_test_scored_once")
    mlflow.set_tag("prior_test_evaluation_before_tuning", "true")
    selected_row_name = "XGBoost tuned" if champion_name == "xgboost_tuned" else "XGBoost default"
    selected_validation_row = validation_comparison.loc[
        validation_comparison.model == selected_row_name
    ].iloc[0]
    mlflow.log_metrics({
        key: float(value)
        for key, value in selected_validation_row.items()
        if key != "model"
    })
    mlflow.log_metric("tuned_relative_gain_fraction", float(tuned_relative_gain))
    log_feature_list()
    log_importance(champion_model, f"{champion_name} feature importance")
    log_xgb_model(champion_model, X_validation_normal.head(5))
    mlflow.log_artifact(str(bundle_path), artifact_path="bundle")
    mlflow.log_artifact(str(validation_path), artifact_path="evaluation")

    # Score the selected champion once on the held-out test partition.
    champion_test_predictions = from_target(champion_model.predict(X_test))
    champion_test_metrics = metrics_for(test, champion_test_predictions, "test")
    mlflow.log_metrics(champion_test_metrics)
    mlflow.log_dict({
        "test_set_previously_scored_before_tuning": True,
        "prior_test_metrics_data_rows": 440972,
        "prior_test_metrics_file": "notebooks/reports/duration_test_metrics.csv",
        "this_notebook_test_scoring_count": 1,
        "champion": champion_name,
        "metrics": champion_test_metrics,
    }, "test_evaluation_disclosure.json")
    champion_run_id = champion_run.info.run_id

test_table = pd.DataFrame([{"model": champion_name, **champion_test_metrics}])
test_path = REPORTS / "availability_duration_champion_test_metrics.csv"
test_table.to_csv(test_path, index=False)

model_uri = f"runs:/{champion_run_id}/model"
registered_version = mlflow.register_model(model_uri=model_uri, name=MODEL_NAME)
MlflowClient().set_registered_model_alias(
    MODEL_NAME, "champion", str(registered_version.version)
)
registered_model_uri = f"models:/{MODEL_NAME}@champion"
handoff = {
    "registered_model_uri": registered_model_uri,
    "champion_run_id": champion_run_id,
    "tracking_uri": TRACKING_URI,
    "experiment_name": EXPERIMENT_NAME,
    "bundle_path": str(bundle_path),
    "data_version": DATA_VERSION,
    "cleaning_count_reconciliation": str(
        TEAM_DIR / "reports" / "tables" / "cleaning_count_reconciliation.csv"
    ),
    "prior_test_scoring_disclosed": True,
    "prior_test_metrics_data_rows": 440972,
    "prior_test_metrics_file": "notebooks/reports/duration_test_metrics.csv",
    "validation_comparison_file": str(validation_path),
    "champion_test_metrics_file": str(test_path),
}
handoff_path = REPORTS / "availability_duration_mlflow_handoff.json"
handoff_path.write_text(json.dumps(handoff, indent=2), encoding="utf-8")
print("Champion run:", champion_run_id)
print("Registered model:", registered_model_uri)
print("Bundle:", bundle_path)
print(test_table.to_string(index=False))

C:\Users\mariam\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


Registered model 'availability_duration' already exists. Creating a new version of this model...
2026/10/28 04:14:32 WARNING mlflow.tracking._model_registry.fluent: Run with id 2bbe5e41509e4552b8aef4b1a11cca40 has no artifacts at artifact path 'model', registering model based on models:/m-e6b4f9544704465eaa5ba044a6ea8b16 instead


Champion run: 2bbe5e41509e4552b8aef4b1a11cca40
Registered model: models:/availability_duration@champion
Bundle: c:\Users\mariam\Desktop\SIC AI\ChargeSentinel-Twin\team\mariam_availability\artifacts\availability_duration_bundle_optuna_mlflow.joblib
          model  test_mae_normal  test_rmse_normal  test_median_ae_normal  test_mae_all
xgboost_default        34.948298         83.049481              17.836515      35.17235


Created version '2' of model 'availability_duration'.


## 9. Export report-ready run table and validation chart

The CSV and PNG files summarize the MLflow runs and validation comparison. To open the interactive UI, run mlflow ui --backend-store-uri sqlite:///artifacts/mlflow.db from team/mariam_availability.

In [11]:
client = MlflowClient()
runs = client.search_runs([EXPERIMENT_ID], order_by=["start_time ASC"], max_results=1000)
run_rows = []
for run in runs:
    run_rows.append({
        "run_name": run.data.tags.get("mlflow.runName", ""),
        "run_id": run.info.run_id,
        "status": run.info.status,
        "model_family": run.data.tags.get("model_family", ""),
        "split_role": run.data.tags.get("split_role", ""),
        "data_sha256_12": run.data.tags.get("data_sha256_12", ""),
        "validation_mae_normal": run.data.metrics.get("validation_mae_normal"),
        "validation_rmse_normal": run.data.metrics.get("validation_rmse_normal"),
        "validation_mae_all": run.data.metrics.get("validation_mae_all"),
        "mean_cv_mae_minutes": run.data.metrics.get("mean_cv_mae_minutes"),
    })
runs_table = pd.DataFrame(run_rows)
runs_csv = REPORTS / "availability_mlflow_runs.csv"
runs_table.to_csv(runs_csv, index=False)

table_figure, table_axis = plt.subplots(
    figsize=(15, max(4, 0.32 * min(len(runs_table), 35)))
)
table_axis.axis("off")
visible_runs = runs_table.tail(35)
table_axis.table(
    cellText=visible_runs[[
        "run_name", "model_family", "status", "validation_mae_normal", "mean_cv_mae_minutes"
    ]].fillna("").values,
    colLabels=["Run", "Family", "Status", "Validation MAE (normal)", "Mean CV MAE"],
    loc="center",
)
table_figure.tight_layout()
runs_png = REPORTS / "availability_mlflow_runs.png"
table_figure.savefig(runs_png, dpi=160, bbox_inches="tight")
plt.close(table_figure)

comparison_figure, comparison_axis = plt.subplots(figsize=(9, 5))
validation_comparison.set_index("model")[[
    "validation_mae_normal", "validation_mae_all"
]].plot.bar(ax=comparison_axis)
comparison_axis.set_ylabel("MAE (minutes)")
comparison_axis.set_title("Availability model validation comparison")
comparison_axis.tick_params(axis="x", rotation=20)
comparison_figure.tight_layout()
comparison_png = REPORTS / "availability_validation_comparison.png"
comparison_figure.savefig(comparison_png, dpi=160, bbox_inches="tight")
plt.close(comparison_figure)

with mlflow.start_run(run_name="experiment_report"):
    set_run_tags("experiment_report", split_role="validation_report")
    mlflow.log_artifact(str(runs_csv), artifact_path="report")
    mlflow.log_artifact(str(runs_png), artifact_path="report")
    mlflow.log_artifact(str(comparison_png), artifact_path="report")
    mlflow.log_artifact(str(selection_path), artifact_path="report")

print("MLflow run table:", runs_csv)
print("Validation chart:", comparison_png)

MLflow run table: c:\Users\mariam\Desktop\SIC AI\ChargeSentinel-Twin\team\mariam_availability\reports\availability_mlflow_runs.csv
Validation chart: c:\Users\mariam\Desktop\SIC AI\ChargeSentinel-Twin\team\mariam_availability\reports\availability_validation_comparison.png


## Final Summary

In [12]:
print("BEST OPTUNA PARAMETERS")
print(json.dumps(study.best_params, indent=2))
print("\nVALIDATION COMPARISON")
print(validation_comparison.to_string(index=False))
print(f"\nTuned relative validation MAE gain: {tuned_relative_gain:.2%}")
print("Champion:", champion_name)
print("\nFINAL CHAMPION TEST RESULT")
print(test_table.to_string(index=False))
print("Test disclosure: previously scored in notebook 05 before tuning; scored once here for champion.")
print("Model URI:", registered_model_uri)
print("Champion run ID:", champion_run_id)
print("Bundle path:", bundle_path)
print("Clean data version:", DATA_VERSION)
print("MLflow tracking URI:", TRACKING_URI)


BEST OPTUNA PARAMETERS
{
  "learning_rate": 0.03199769450722887,
  "max_depth": 6,
  "min_child_weight": 4,
  "subsample": 0.9532924675152317,
  "colsample_bytree": 0.539732865426287,
  "reg_lambda": 8.488401811039807,
  "reg_alpha": 0.04728690732525066,
  "gamma": 1.1451040057231605
}

VALIDATION COMPARISON
          model  validation_mae_normal  validation_rmse_normal  validation_median_ae_normal  validation_mae_all
       Baseline              44.107766              109.294218                    19.200000           43.066884
  Random Forest              43.628535              107.146282                    18.379218           40.809935
XGBoost default              43.403724              107.275742                    18.241268           40.692389
  XGBoost tuned              43.397954              107.173239                    18.251621           40.664084

Tuned relative validation MAE gain: 0.01%
Champion: xgboost_default

FINAL CHAMPION TEST RESULT
          model  test_mae_normal 